In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Configuration
source_table = "etl_restapi.silver.earthquake_data"
target_catalog = "etl_restapi"
target_schema = "gold"

# Create gold schema if not exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_catalog}.{target_schema}")

print(f"Gold schema: {target_catalog}.{target_schema}")

Gold schema: etl_restapi.gold


In [0]:
# Dimension Table: Location
# Extract unique location information

df_source = spark.table(source_table)

dim_location = df_source.select(
    F.md5(F.concat_ws("|", 
                      F.coalesce(F.col("place"), F.lit("")),
                      F.col("longitude").cast("string"),
                      F.col("latitude").cast("string"),
                      F.col("depth_km").cast("string")
                     )).alias("location_key"),
    F.col("place"),
    F.col("longitude"),
    F.col("latitude"),
    F.col("depth_km").alias("depth"),
    # Extract country/region from place field
    F.when(F.col("place").contains(","), 
           F.trim(F.split(F.col("place"), ",").getItem(1))
          ).otherwise(F.col("place")).alias("region"),
    F.current_timestamp().alias("created_at")
).distinct()

# Write to gold layer
dim_location.write.mode("overwrite").saveAsTable(f"{target_catalog}.{target_schema}.dim_location")

print(f"✓ Created dim_location with {dim_location.count()} records")
display(dim_location.limit(5))

✓ Created dim_location with 417 records


location_key,place,longitude,latitude,depth,region,created_at
326a6b1988119a62809eb97903559ad9,"16 km NW of Midland, Texas",-102.215,32.089,4.1298,Texas,2026-09-01T15:54:45.997Z
7c11e22c3400b6a9c4802d3d8f0b8047,"4 km NNE of Beaumont, CA",-116.9615,33.9631666666667,8.3,CA,2026-09-01T15:54:45.997Z
94a8687cff42dfd7a758c5b95c31d818,"10 km N of Midland, Texas",-102.074,32.091,8.0258,Texas,2026-09-01T15:54:45.997Z
886ad37927460ffaca562b95c2d4b372,"12 km WSW of Salton City, CA",-116.0665,33.2428333333333,4.34,CA,2026-09-01T15:54:45.997Z
823124af435355494c2367864c8f12f8,"3 km ENE of Medford, Oklahoma",-97.70416667,36.82016667,7.16,Oklahoma,2026-09-01T15:54:45.997Z


In [0]:
# Dimension Table: Earthquake Type
# Extract unique type information

dim_earthquake_type = df_source.select(
    F.md5(F.concat_ws("|",
                      F.coalesce(F.col("event_type"), F.lit("")),
                      F.coalesce(F.col("magnitude_type"), F.lit("")),
                      F.coalesce(F.col("status"), F.lit(""))
                     )).alias("earthquake_type_key"),
    F.col("event_type").alias("event_type"),
    F.col("magnitude_type").alias("magnitude_type"),
    F.col("status"),
    F.current_timestamp().alias("created_at")
).distinct()

# Write to gold layer
dim_earthquake_type.write.mode("overwrite").saveAsTable(f"{target_catalog}.{target_schema}.dim_earthquake_type")

print(f"✓ Created dim_earthquake_type with {dim_earthquake_type.count()} records")
display(dim_earthquake_type.limit(5))

✓ Created dim_earthquake_type with 8 records


earthquake_type_key,event_type,magnitude_type,status,created_at
a7116183718ef07ab2f027bf3bce799d,earthquake,ml,automatic,2026-09-01T15:54:59.867Z
8e910cba1d32c2d885d24ea6a372f6bb,earthquake,ml,reviewed,2026-09-01T15:54:59.867Z
d20bc02abd7c92e072ce14740dc64308,earthquake,md,automatic,2026-09-01T15:54:59.867Z
67e7477bada8c3ffac1a60cdeb8a15df,earthquake,mb,reviewed,2026-09-01T15:54:59.867Z
e7cc6b06f39671f5313f65e14bbc2518,earthquake,md,reviewed,2026-09-01T15:54:59.867Z


In [0]:
# Dimension Table: Network
# Extract unique network/source information

dim_network = df_source.select(
    F.md5(F.concat_ws("|",
                      F.coalesce(F.col("network"), F.lit("")),
                      F.coalesce(F.col("event_code"), F.lit("")),
                      F.coalesce(F.col("sources"), F.lit(""))
                     )).alias("network_key"),
    F.col("network").alias("network_code"),
    F.col("event_code").alias("event_code"),
    F.col("sources"),
    F.current_timestamp().alias("created_at")
).distinct()

# Write to gold layer
dim_network.write.mode("overwrite").saveAsTable(f"{target_catalog}.{target_schema}.dim_network")

print(f"✓ Created dim_network with {dim_network.count()} records")
display(dim_network.limit(5))

✓ Created dim_network with 417 records


network_key,network_code,event_code,sources,created_at
bbf78ae57c08402acaf9c5980ee5d6d9,tx,2026qwdcwr,",tx,",2026-09-01T15:55:04.931Z
44dd22785a006a72a64ffbd9d6b92293,ci,41537544,",ci,",2026-09-01T15:55:04.931Z
701806c6d35ddc840e222769edada97e,tx,2026qwcnkr,",tx,",2026-09-01T15:55:04.931Z
1de2ea14b68d2bf57544698284345204,ci,41537536,",ci,",2026-09-01T15:55:04.931Z
9f99300fb10ff9bbae23a5aa2a86eb88,ok,2026quax,",ok,",2026-09-01T15:55:04.931Z


In [0]:
# Dimension Table: Alert
# Extract unique alert information

dim_alert = df_source.select(
    F.md5(F.concat_ws("|",
                      F.coalesce(F.col("alert_level"), F.lit("")),
                      F.col("tsunami_flag").cast("string")
                     )).alias("alert_key"),
    F.col("alert_level").alias("alert_level"),
    F.col("tsunami_flag").cast("int").alias("tsunami_flag"),
    F.when(F.col("tsunami_flag") == 1, "Yes").otherwise("No").alias("tsunami_risk"),
    F.current_timestamp().alias("created_at")
).distinct()

# Write to gold layer
dim_alert.write.mode("overwrite").saveAsTable(f"{target_catalog}.{target_schema}.dim_alert")

print(f"✓ Created dim_alert with {dim_alert.count()} records")
display(dim_alert.limit(5))

✓ Created dim_alert with 2 records


alert_key,alert_level,tsunami_flag,tsunami_risk,created_at
ef7323af4582f50d5e939a198222d68c,null,0,No,2026-09-01T15:55:11.164Z
fd810b83bcca406955c9940ff0bf88ba,green,0,No,2026-09-01T15:55:11.164Z


In [0]:
# Dimension Table: Date
# Extract date parts for time-based analysis

dim_date = df_source.select(
    F.date_format(F.col("event_time"), "yyyyMMdd").alias("date_key"),
    F.to_date(F.col("event_time")).alias("date"),
    F.year(F.col("event_time")).alias("year"),
    F.month(F.col("event_time")).alias("month"),
    F.dayofmonth(F.col("event_time")).alias("day"),
    F.dayofweek(F.col("event_time")).alias("day_of_week"),
    F.date_format(F.col("event_time"), "EEEE").alias("day_name"),
    F.weekofyear(F.col("event_time")).alias("week_of_year"),
    F.quarter(F.col("event_time")).alias("quarter"),
    F.date_format(F.col("event_time"), "MMMM").alias("month_name"),
    F.current_timestamp().alias("created_at")
).distinct().orderBy("date")

# Write to gold layer
dim_date.write.mode("overwrite").saveAsTable(f"{target_catalog}.{target_schema}.dim_date")

print(f"✓ Created dim_date with {dim_date.count()} records")
display(dim_date.limit(5))

✓ Created dim_date with 4 records


date_key,date,year,month,day,day_of_week,day_name,week_of_year,quarter,month_name,created_at
20260826,2026-08-26,2026,8,26,4,Wednesday,35,3,August,2026-09-01T15:55:17.125Z
20260827,2026-08-27,2026,8,27,5,Thursday,35,3,August,2026-09-01T15:55:17.125Z
20260831,2026-08-31,2026,8,31,2,Monday,36,3,August,2026-09-01T15:55:17.125Z
20260901,2026-09-01,2026,9,1,3,Tuesday,36,3,September,2026-09-01T15:55:17.125Z


In [0]:
# Fact Table: Earthquake
# Central fact table with measurements and foreign keys

fact_earthquake = df_source.select(
    F.col("event_id").alias("earthquake_id"),
    
    # Foreign Keys
    F.date_format(F.col("event_time"), "yyyyMMdd").alias("date_key"),
    F.md5(F.concat_ws("|", 
                      F.coalesce(F.col("place"), F.lit("")),
                      F.col("longitude").cast("string"),
                      F.col("latitude").cast("string"),
                      F.col("depth_km").cast("string")
                     )).alias("location_key"),
    F.md5(F.concat_ws("|",
                      F.coalesce(F.col("event_type"), F.lit("")),
                      F.coalesce(F.col("magnitude_type"), F.lit("")),
                      F.coalesce(F.col("status"), F.lit(""))
                     )).alias("earthquake_type_key"),
    F.md5(F.concat_ws("|",
                      F.coalesce(F.col("network"), F.lit("")),
                      F.coalesce(F.col("event_code"), F.lit("")),
                      F.coalesce(F.col("sources"), F.lit(""))
                     )).alias("network_key"),
    F.md5(F.concat_ws("|",
                      F.coalesce(F.col("alert_level"), F.lit("")),
                      F.col("tsunami_flag").cast("string")
                     )).alias("alert_key"),
    
    # Time attributes
    F.col("event_time").alias("event_timestamp"),
    
    # Measures
    F.col("magnitude").alias("magnitude"),
    F.col("significance").alias("significance"),
    F.col("felt_reports").alias("felt_reports"),
    F.col("cdi"),
    F.col("mmi"),
    F.col("nst").alias("num_stations"),
    F.col("dmin"),
    F.col("rms"),
    F.col("gap"),
    
    # Additional attributes
    F.col("title"),
    F.col("event_url").alias("url"),
    F.col("detail_url").alias("detail"),
    F.col("ids"),
    F.col("types"),
    
    # Audit fields
    F.col("ingestion_time").alias("source_load_timestamp"),
    F.current_timestamp().alias("created_at")
)

# Write to gold layer
fact_earthquake.write.mode("overwrite").saveAsTable(f"{target_catalog}.{target_schema}.fact_earthquake")

print(f"✓ Created fact_earthquake with {fact_earthquake.count()} records")
display(fact_earthquake.limit(5))

✓ Created fact_earthquake with 417 records


earthquake_id,date_key,location_key,earthquake_type_key,network_key,alert_key,event_timestamp,magnitude,significance,felt_reports,cdi,mmi,num_stations,dmin,rms,gap,title,url,detail,ids,types,source_load_timestamp,created_at
tx2026qwdcwr,20260827,326a6b1988119a62809eb97903559ad9,a7116183718ef07ab2f027bf3bce799d,bbf78ae57c08402acaf9c5980ee5d6d9,ef7323af4582f50d5e939a198222d68c,2026-08-27T16:14:05.000Z,1.3,26,null,null,null,43,0.0,0.4,56.0,"M 1.3 - 16 km NW of Midland, Texas",https://earthquake.usgs.gov/earthquakes/eventpage/tx2026qwdcwr,https://earthquake.usgs.gov/earthquakes/feed/v1.0/detail/tx2026qwdcwr.geojson,",tx2026qwdcwr,",",origin,phase-data,",2026-09-01T15:52:41.141Z,2026-09-01T15:55:24.126Z
ci41537544,20260827,7c11e22c3400b6a9c4802d3d8f0b8047,a7116183718ef07ab2f027bf3bce799d,44dd22785a006a72a64ffbd9d6b92293,ef7323af4582f50d5e939a198222d68c,2026-08-27T15:57:47.000Z,1.29,26,null,null,null,61,0.1318,0.15,23.0,"M 1.3 - 4 km NNE of Beaumont, CA",https://earthquake.usgs.gov/earthquakes/eventpage/ci41537544,https://earthquake.usgs.gov/earthquakes/feed/v1.0/detail/ci41537544.geojson,",ci41537544,",",focal-mechanism,nearby-cities,origin,phase-data,",2026-09-01T15:52:41.141Z,2026-09-01T15:55:24.126Z
tx2026qwcnkr,20260827,94a8687cff42dfd7a758c5b95c31d818,a7116183718ef07ab2f027bf3bce799d,701806c6d35ddc840e222769edada97e,ef7323af4582f50d5e939a198222d68c,2026-08-27T15:56:09.000Z,1.5,35,null,null,null,14,0.0,0.5,115.0,"M 1.5 - 10 km N of Midland, Texas",https://earthquake.usgs.gov/earthquakes/eventpage/tx2026qwcnkr,https://earthquake.usgs.gov/earthquakes/feed/v1.0/detail/tx2026qwcnkr.geojson,",tx2026qwcnkr,",",origin,phase-data,",2026-09-01T15:52:41.141Z,2026-09-01T15:55:24.126Z
ci41537536,20260827,886ad37927460ffaca562b95c2d4b372,a7116183718ef07ab2f027bf3bce799d,1de2ea14b68d2bf57544698284345204,ef7323af4582f50d5e939a198222d68c,2026-08-27T15:48:01.000Z,0.98,15,null,null,null,38,0.07712,0.23,55.0,"M 1.0 - 12 km WSW of Salton City, CA",https://earthquake.usgs.gov/earthquakes/eventpage/ci41537536,https://earthquake.usgs.gov/earthquakes/feed/v1.0/detail/ci41537536.geojson,",ci41537536,",",nearby-cities,origin,phase-data,",2026-09-01T15:52:41.141Z,2026-09-01T15:55:24.126Z
ok2026quax,20260827,823124af435355494c2367864c8f12f8,8e910cba1d32c2d885d24ea6a372f6bb,9f99300fb10ff9bbae23a5aa2a86eb88,ef7323af4582f50d5e939a198222d68c,2026-08-27T15:45:19.000Z,1.3,26,null,null,null,36,0.1286747496,0.16,79.0,"M 1.3 - 3 km ENE of Medford, Oklahoma",https://earthquake.usgs.gov/earthquakes/eventpage/ok2026quax,https://earthquake.usgs.gov/earthquakes/feed/v1.0/detail/ok2026quax.geojson,",ok2026quax,",",origin,phase-data,",2026-09-01T15:52:41.141Z,2026-09-01T15:55:24.126Z


In [0]:
# Verify the star schema by joining fact with dimensions

verification_query = f"""
SELECT 
    f.earthquake_id,
    d.date,
    d.year,
    d.month_name,
    l.place,
    l.region,
    l.latitude,
    l.longitude,
    f.magnitude,
    et.event_type,
    et.magnitude_type,
    et.status,
    a.alert_level,
    a.tsunami_risk,
    n.network_code,
    f.significance,
    f.felt_reports
FROM {target_catalog}.{target_schema}.fact_earthquake f
INNER JOIN {target_catalog}.{target_schema}.dim_date d ON f.date_key = d.date_key
INNER JOIN {target_catalog}.{target_schema}.dim_location l ON f.location_key = l.location_key
INNER JOIN {target_catalog}.{target_schema}.dim_earthquake_type et ON f.earthquake_type_key = et.earthquake_type_key
INNER JOIN {target_catalog}.{target_schema}.dim_alert a ON f.alert_key = a.alert_key
INNER JOIN {target_catalog}.{target_schema}.dim_network n ON f.network_key = n.network_key
ORDER BY f.event_timestamp DESC
LIMIT 10
"""

result_df = spark.sql(verification_query)
print("\n✓ Star Schema Verification - Sample Records:")
display(result_df)


✓ Star Schema Verification - Sample Records:


earthquake_id,date,year,month_name,place,region,latitude,longitude,magnitude,event_type,magnitude_type,status,alert_level,tsunami_risk,network_code,significance,felt_reports
nc75428242,2026-09-01,2026,September,"16 km NE of Point Arena, CA",CA,39.0103340148926,-123.564834594727,1.22,earthquake,md,automatic,null,No,nc,23,null
nc75428237,2026-09-01,2026,September,"6 km WNW of Cobb, CA",CA,38.8373336791992,-122.785835266113,0.39,earthquake,md,automatic,null,No,nc,2,null
ci41540152,2026-09-01,2026,September,"20 km NW of Progreso, B.C., MX",B.C.,32.6905,-115.752166666667,2.3,earthquake,ml,automatic,null,No,ci,81,null
nc75428212,2026-09-01,2026,September,"7 km NW of The Geysers, CA",CA,38.8139991760254,-122.8193359375,1.0,earthquake,md,automatic,null,No,nc,15,null
tx2026rfeqvz,2026-09-01,2026,September,"58 km S of Whites City, New Mexico",New Mexico,31.654,-104.423,2.2,earthquake,ml,reviewed,null,No,tx,74,null
nc75428207,2026-09-01,2026,September,"9 km NW of The Geysers, CA",CA,38.833667755127,-122.827163696289,0.68,earthquake,md,automatic,null,No,nc,7,null
nc75428192,2026-09-01,2026,September,"3 km NNW of Redwood Valley, CA",CA,39.2890014648438,-123.214332580566,1.26,earthquake,md,automatic,null,No,nc,24,null
aka2026rhymwb,2026-09-01,2026,September,"72 km WNW of Beluga, Alaska",Alaska,61.377,-152.335,0.9,earthquake,ml,automatic,null,No,ak,12,null
aka2026rhykoi,2026-09-01,2026,September,"23 km NNW of Petersville, Alaska",Alaska,62.68,-150.993,1.8,earthquake,ml,automatic,null,No,ak,50,null
tx2026rfeetf,2026-09-01,2026,September,"12 km SW of Falls City, Texas",Texas,28.897,-98.102,3.6,earthquake,ml,reviewed,null,No,tx,200,2


In [0]:
# Summary statistics for the gold layer

summary_query = f"""
SELECT 
    'dim_location' as table_name,
    COUNT(*) as record_count
FROM {target_catalog}.{target_schema}.dim_location

UNION ALL

SELECT 
    'dim_earthquake_type' as table_name,
    COUNT(*) as record_count
FROM {target_catalog}.{target_schema}.dim_earthquake_type

UNION ALL

SELECT 
    'dim_network' as table_name,
    COUNT(*) as record_count
FROM {target_catalog}.{target_schema}.dim_network

UNION ALL

SELECT 
    'dim_alert' as table_name,
    COUNT(*) as record_count
FROM {target_catalog}.{target_schema}.dim_alert

UNION ALL

SELECT 
    'dim_date' as table_name,
    COUNT(*) as record_count
FROM {target_catalog}.{target_schema}.dim_date

UNION ALL

SELECT 
    'fact_earthquake' as table_name,
    COUNT(*) as record_count
FROM {target_catalog}.{target_schema}.fact_earthquake
"""

print("\n📊 Gold Layer Summary:")
print("="*50)
display(spark.sql(summary_query))

print("\n✅ Gold layer star schema created successfully!")
print(f"\nCatalog: {target_catalog}")
print(f"Schema: {target_schema}")
print("\nTables created:")
print("  • dim_location")
print("  • dim_earthquake_type")
print("  • dim_network") 
print("  • dim_alert")
print("  • dim_date")
print("  • fact_earthquake")


📊 Gold Layer Summary:


table_name,record_count
dim_location,417
dim_earthquake_type,8
dim_network,417
dim_alert,2
dim_date,4
fact_earthquake,417



✅ Gold layer star schema created successfully!

Catalog: etl_restapi
Schema: gold

Tables created:
  • dim_location
  • dim_earthquake_type
  • dim_network
  • dim_alert
  • dim_date
  • fact_earthquake
